In [11]:
import os
import json
import math
import random
import zipfile
import string
from typing import List, Dict, Tuple, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

TOKENIZER_PATH = "/kaggle/input/datasets/brindafadadu/cs781-arch2-phase1/custom_bpe.json"
CHECKPOINT_PATH = "/kaggle/input/datasets/brindafadadu/cs781-arch2-phase1/decoder_lm.pt"

MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
HEAD_LR = 1e-3
EPOCHS = 12
USE_VAL=True

# LoRA Hyperparameters
LORA_RANK = 32
LORA_ALPHA = 64.0
LORA_DROPOUT = 0.05

LABEL_NAMES = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]

ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx

NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100

print(f"Device: {DEVICE} | Ready for Experiment 2 (Arch-2 RoPE + LoRA)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda | Ready for Experiment 2 (Arch-2 RoPE + LoRA)


In [2]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

    def _word_to_symbols(self, word):
        return list(word.encode("utf-8").decode("latin-1")) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode_with_exact_offsets(self, text: str) -> Tuple[List[int], List[Tuple[int, int]]]:
        """
        Calculates EXACT character-level [start, end) offsets for every single subword token.
        """
        tokens = []
        offsets = []
        i = 0
        n = len(text)

        while i < n:
            while i < n and text[i].isspace():
                i += 1
            if i >= n:
                break
            word_start = i
            while i < n and not text[i].isspace():
                i += 1
            word_end = i
            word = text[word_start:word_end]

            subwords = self._apply_merges_to_word(word)
            curr_pos = word_start

            for sym in subwords:
                clean_sym = sym.replace(END_OF_WORD, "")
                try:
                    sym_chars = clean_sym.encode("latin-1").decode("utf-8", errors="replace")
                except Exception:
                    sym_chars = clean_sym

                sub_len = len(sym_chars)
                sub_start = curr_pos
                sub_end = min(word_end, curr_pos + sub_len)
                curr_pos = sub_end

                t_id = self.stoi.get(sym, self.stoi.get("<unk>", 1))
                tokens.append(t_id)
                offsets.append((sub_start, sub_end))

        return tokens, offsets

# Load tokenizer
tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer | Vocab size: {tokenizer.vocab_size}")

Loaded tokenizer | Vocab size: 8000


In [10]:
# EXACT ARCHITECTURE 2 (RoPE + QK-Norm + RMSNorm + SwiGLU) + STANDARD LoRA
from dataclasses import dataclass

@dataclass
class Arch2Config:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2560
    max_seq_len: int = 512
    dropout: float = 0.1
    rope_theta: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048, theta: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        return self.cos_cached[:seq_len].to(x.dtype), self.sin_cached[:seq_len].to(x.dtype)

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos.unsqueeze(0).unsqueeze(1)
    sin = sin.unsqueeze(0).unsqueeze(1)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRALinear(nn.Module):
    """Standard LoRA Adapter (Hu et al., 2021)"""
    def __init__(self, original_linear: nn.Linear, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05):
        super().__init__()
        self.original_linear = original_linear
        self.rank = rank
        self.scaling = alpha / rank
        self.original_linear.weight.requires_grad = False

        in_f = original_linear.in_features
        out_f = original_linear.out_features
        self.lora_dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)

        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.original_linear(x) + self.lora_B(self.lora_A(self.lora_dropout(x))) * self.scaling

class RoPECausalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        z = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        z = z.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(z)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class RoPEBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = RoPECausalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLUFeedForward(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ffn(self.norm2(x))
        return x

class Arch2LNERModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 768, n_layers: int = 12,
                 n_heads: int = 12, d_ff: int = 2560, max_seq_len: int = 512,
                 num_classes: int = NUM_CLASSES, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.rope = RotaryEmbedding(d_model // n_heads, max_seq_len)
        self.blocks = nn.ModuleList([
            RoPEBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        
        # Token Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos, sin = self.rope(x, T)
        for block in self.blocks:
            x = block(x, cos, sin)
        x = self.norm_f(x)
        return self.classifier(x)

    def apply_lora(self, rank: int = 16, alpha: float = 32.0, dropout: float = 0.05, device=DEVICE):
        for param in self.parameters():
            param.requires_grad = False

        for block in self.blocks:
            block.attn.q_proj = LoRALinear(block.attn.q_proj, rank=rank, alpha=alpha, dropout=dropout).to(device)
            block.attn.v_proj = LoRALinear(block.attn.v_proj, rank=rank, alpha=alpha, dropout=dropout).to(device)

        for param in self.classifier.parameters():
            param.requires_grad = True

# Load Model & Weights
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

d_model_ckpt = state_dict["tok_emb.weight"].shape[1]
vocab_size_ckpt = state_dict["tok_emb.weight"].shape[0]
block_indices = [int(k.split(".")[1]) for k in state_dict.keys() if k.startswith("blocks.") and k.split(".")[1].isdigit()]
n_layers_ckpt = max(block_indices) + 1 if block_indices else 12
n_heads = 8 if d_model_ckpt == 640 else 12

model = Arch2LNERModel(vocab_size=vocab_size_ckpt, d_model=d_model_ckpt, n_layers=n_layers_ckpt, n_heads=n_heads).to(DEVICE)
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("lm_head") and not k.startswith("head")}
missing, unexpected = model.load_state_dict(backbone_dict, strict=False)
print("MISSING   :", len(missing), missing[:8])
print("UNEXPECTED:", len(unexpected), unexpected[:8])
assert all(m.startswith("classifier.") for m in missing), "Backbone weights did NOT load - fix key names before continuing!"

def apply_lora_all(model, rank, alpha, dropout, device):
    for p in model.parameters():
        p.requires_grad = False
    for b in model.blocks:
        for name in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            setattr(b.attn, name, LoRALinear(getattr(b.attn, name), rank, alpha, dropout).to(device))
        for name in ["gate_proj", "up_proj", "down_proj"]:
            setattr(b.ffn, name, LoRALinear(getattr(b.ffn, name), rank, alpha, dropout).to(device))
    for p in model.classifier.parameters():
        p.requires_grad = True
    for p in model.norm_f.parameters():
        p.requires_grad = True

apply_lora_all(model, LORA_RANK, LORA_ALPHA, LORA_DROPOUT, DEVICE)
model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Arch-2 + LoRA(all linear) | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

MISSING   : 6 ['classifier.0.weight', 'classifier.0.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias']
UNEXPECTED: 0 []
Arch-2 + LoRA(all linear) | Trainable: 6,805,273 (6.07%)


In [13]:
def download_phase2_file(filename: str) -> str:
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Downloading dataset files...")
f1_text_path = download_phase2_file("Phase_2_LNER/Train/fold1-text.json")
f1_gold_path = download_phase2_file("Phase_2_LNER/Train/fold1-gold.json")
f2_text_path = download_phase2_file("Phase_2_LNER/Train/fold2-text.json")
f2_gold_path = download_phase2_file("Phase_2_LNER/Train/fold2-gold.json")
f3_text_path = download_phase2_file("Phase_2_LNER/Test/fold3-text.json")

with open(f1_text_path, "r", encoding="utf-8") as f: train_texts = json.load(f)
with open(f1_gold_path, "r", encoding="utf-8") as f: train_golds = json.load(f)
with open(f2_text_path, "r", encoding="utf-8") as f: val_texts = json.load(f)
with open(f2_gold_path, "r", encoding="utf-8") as f: val_golds = json.load(f)
with open(f3_text_path, "r", encoding="utf-8") as f: test_texts = json.load(f)

# Combine folds 1 and 2 to maximize training supervision
if USE_VAL:
    tr_texts, tr_golds = train_texts, train_golds
else:
    tr_texts = {**train_texts, **val_texts}
    tr_golds = {**train_golds, **val_golds}

print(f"Training docs: {len(tr_texts)} | USE_VAL={USE_VAL}")

class ExactLNERDataset(Dataset):
    def __init__(self, docs: Dict[str, str], golds: Dict[str, List[Any]], tokenizer: ByteBPETokenizer,
                 max_len: int = MAX_SEQ_LEN, stride: int = STRIDE):
        self.samples = []
        for doc_id, text in docs.items():
            token_ids, offsets = tokenizer.encode_with_exact_offsets(text)

            token_labels = [0] * len(token_ids)
            if doc_id in golds:
                for annot in golds[doc_id]:
                    g_start, g_end, g_lbl = annot[0], annot[1], annot[2]
                    first_matched = False
                    for idx, (t_s, t_e) in enumerate(offsets):
                        if max(t_s, g_start) < min(t_e, g_end):
                            if not first_matched:
                                token_labels[idx] = LABEL2ID.get(f"B-{g_lbl}", 0)
                                first_matched = True
                            else:
                                token_labels[idx] = LABEL2ID.get(f"I-{g_lbl}", 0)

            for start_idx in range(0, max(1, len(token_ids)), max_len - stride):
                end_idx = min(len(token_ids), start_idx + max_len)
                c_ids = token_ids[start_idx:end_idx]
                c_lbls = token_labels[start_idx:end_idx]

                pad_len = max_len - len(c_ids)
                padded_ids = c_ids + [0] * pad_len
                padded_lbls = c_lbls + [IGNORE_INDEX] * pad_len

                self.samples.append({
                    "input_ids": torch.tensor(padded_ids, dtype=torch.long),
                    "labels": torch.tensor(padded_lbls, dtype=torch.long)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ExactLNERDataset(tr_texts, tr_golds, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training sliding windows: {len(train_ds)}")

Training docs: 35 | USE_VAL=True
Total training sliding windows: 667


In [16]:
import numpy as np

# --- boundary cleanup: set these after reading the Cell 2 inspection output ---
STRIP_LEAD  = " \t\n\r"
STRIP_TRAIL = " \t\n\r,;:"

def clean_span(text, start, end, label):
    while start < end and text[start] in STRIP_LEAD:
        start += 1
    while end > start and text[end - 1] in STRIP_TRAIL:
        end -= 1
    return start, end, label

def decode_spans_exact(doc_text, pred_ids, offsets):
    spans, cur_l, cs, ce = [], None, -1, -1
    def flush():
        if cur_l is not None:
            s, e, l = clean_span(doc_text, cs, ce, cur_l)
            if e > s: spans.append([s, e, l])
    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        tag = ID2LABEL.get(int(t_id), "O")
        if tag.startswith("B-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        elif tag.startswith("I-") and cur_l == tag[2:]:
            ce = e_c
        elif tag.startswith("I-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        else:
            flush(); cur_l = None
    flush()
    seen, out = set(), []
    for s in spans:
        k = tuple(s)
        if k not in seen:
            seen.add(k); out.append(s)
    return out

# --- BIO-constrained Viterbi ---
NEG = -1e9
ALLOWED = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=bool)   # [prev, cur]
for cur in range(NUM_CLASSES):
    if ID2LABEL[cur].startswith("I-"):
        for prev in range(NUM_CLASSES):
            ptag = ID2LABEL[prev]
            ALLOWED[prev, cur] = (ptag != "O") and (ptag[2:] == ID2LABEL[cur][2:])
START_OK = np.array([not ID2LABEL[c].startswith("I-") for c in range(NUM_CLASSES)])
TRANS = np.where(ALLOWED, 0.0, NEG)

def viterbi(logp):
    T, C = logp.shape
    score = logp[0].copy(); score[~START_OK] = NEG
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

@torch.no_grad()
def predict_doc(text):
    ids, offs = tokenizer.encode_with_exact_offsets(text)
    if not ids:
        return []
    acc = torch.zeros(len(ids), NUM_CLASSES); cnt = torch.zeros(len(ids), 1)
    for s in range(0, max(1, len(ids)), MAX_SEQ_LEN - STRIDE):
        e = min(len(ids), s + MAX_SEQ_LEN)
        x = torch.tensor([ids[s:e]], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            lp = F.log_softmax(model(x)[0].float(), dim=-1).cpu()
        acc[s:e] += lp; cnt[s:e] += 1
        if e == len(ids):
            break
    path = viterbi((acc / cnt).numpy())
    return decode_spans_exact(text, path, offs)

def predict_corpus(texts):
    model.eval()
    return {d: predict_doc(t) for d, t in texts.items()}

def strict_macro_f1(pred, gold, labels=LABEL_NAMES):
    f1s = []
    for lbl in labels:
        tp = fp = fn = 0
        for d in gold:
            g = {(a[0], a[1]) for a in gold[d] if a[2] == lbl}
            p = {(a[0], a[1]) for a in pred.get(d, []) if a[2] == lbl}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s), dict(zip(labels, f1s))

In [17]:
# class weights (O = 1, rare tags up to 20x)
counts = torch.zeros(NUM_CLASSES)
for s in train_ds.samples:
    l = s["labels"]; l = l[l != IGNORE_INDEX]
    counts += torch.bincount(l, minlength=NUM_CLASSES).float()
w = (counts.sum() / counts.clamp(min=1)) ** 0.5
w = (w / w[0]).clamp(max=20.0)
print("O fraction:", (counts[0] / counts.sum()).item())
criterion = nn.CrossEntropyLoss(weight=w.to(DEVICE), ignore_index=IGNORE_INDEX)

# optimizer with separate head LR
head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
other_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": LEARNING_RATE},
    {"params": head_params,  "lr": HEAD_LR},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))
def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    prog = (s - warmup) / max(1, total_steps - warmup)
    return 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_f1, best_state = -1.0, None
EVAL_EVERY = 2

for epoch in range(EPOCHS):
    model.train()
    running, n_log = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids)
        loss = criterion(logits.float().view(-1, NUM_CLASSES), labels.view(-1))
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        running += loss.item(); n_log += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{step+1}/{len(train_loader)}] Loss: {running/n_log:.4f}")
            running, n_log = 0.0, 0

    if USE_VAL and ((epoch + 1) % EVAL_EVERY == 0 or epoch + 1 == EPOCHS):
        preds = predict_corpus(val_texts)
        f1, per = strict_macro_f1(preds, val_golds)
        print(f"==> Epoch {epoch+1} | VAL strict macro-F1: {f1:.4f}")
        print({k: round(v, 3) for k, v in per.items()})
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in model.named_parameters() if v.requires_grad}
            best_epoch = epoch + 1

if USE_VAL and best_state is not None:
    model.load_state_dict(best_state, strict=False)
    print(f"Restored best epoch {best_epoch} (val F1 {best_f1:.4f})")
print("✓ Training completed!")

O fraction: 0.9017741680145264
Epoch [1/12] Step [100/167] Loss: 0.6662
Epoch [2/12] Step [100/167] Loss: 0.4812
==> Epoch 2 | VAL strict macro-F1: 0.1883
{'APP': 0.058, 'RESP': 0.004, 'A.COUNSEL': 0.04, 'R.COUNSEL': 0.286, 'JUDGE': 0.139, 'WIT': 0.035, 'AUTH': 0.193, 'COURT': 0.484, 'STAT': 0.302, 'PREC': 0.063, 'DATE': 0.527, 'CASENO': 0.13}
Epoch [3/12] Step [100/167] Loss: 0.3948
Epoch [4/12] Step [100/167] Loss: 0.3018
==> Epoch 4 | VAL strict macro-F1: 0.2011
{'APP': 0.024, 'RESP': 0.004, 'A.COUNSEL': 0.014, 'R.COUNSEL': 0.305, 'JUDGE': 0.178, 'WIT': 0.035, 'AUTH': 0.139, 'COURT': 0.553, 'STAT': 0.317, 'PREC': 0.086, 'DATE': 0.589, 'CASENO': 0.169}
Epoch [5/12] Step [100/167] Loss: 0.2365
Epoch [6/12] Step [100/167] Loss: 0.1984
==> Epoch 6 | VAL strict macro-F1: 0.2098
{'APP': 0.038, 'RESP': 0.01, 'A.COUNSEL': 0.255, 'R.COUNSEL': 0.176, 'JUDGE': 0.149, 'WIT': 0.008, 'AUTH': 0.194, 'COURT': 0.518, 'STAT': 0.365, 'PREC': 0.09, 'DATE': 0.561, 'CASENO': 0.154}
Epoch [7/12] Step [100

In [18]:
test_predictions = predict_corpus(test_texts)
test_predictions = {d: [[int(a), int(b), str(l)] for a, b, l in sp] for d, sp in test_predictions.items()}

output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(test_predictions)); f.flush(); os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")
print(f"✓ Submission ready: {output_zip} ({os.path.getsize(output_zip):,} bytes)")

✓ Submission ready: /kaggle/working/submission.zip (46,135 bytes)


In [69]:
# LOCAL STRICT EVALUATOR (EXACT CODABENCH PROTOCOL)

def evaluate_predictions_strictly(gold_data: Dict[str, List[Any]], pred_data: Dict[str, List[Any]]):
    """
    Computes Strict Macro-F1, Precision, and Recall across all 12 legal entity classes.
    An entity is strictly correct ONLY if [start, end, label] match exactly.
    """
    ALL_LABELS = [
        "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
        "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
    ]
    
    # Track True Positives, False Positives, False Negatives per label
    tp_per_label = {lbl: 0 for lbl in ALL_LABELS}
    fp_per_label = {lbl: 0 for lbl in ALL_LABELS}
    fn_per_label = {lbl: 0 for lbl in ALL_LABELS}

    for doc_id, gold_spans in gold_data.items():
        # Represent gold spans as a set of tuples: {(start, end, label), ...}
        gold_set = set()
        for sp in gold_spans:
            if len(sp) >= 3 and sp[2] in ALL_LABELS:
                gold_set.add((int(sp[0]), int(sp[1]), str(sp[2]).strip()))
                
        # Represent predicted spans as a set: {(start, end, label), ...}
        pred_set = set()
        if doc_id in pred_data:
            for sp in pred_data[doc_id]:
                if len(sp) >= 3 and sp[2] in ALL_LABELS:
                    pred_set.add((int(sp[0]), int(sp[1]), str(sp[2]).strip()))

        # Matches = Intersection
        matched = gold_set.intersection(pred_set)
        
        for sp in matched:
            tp_per_label[sp[2]] += 1
            
        for sp in pred_set - gold_set:
            fp_per_label[sp[2]] += 1
            
        for sp in gold_set - pred_set:
            fn_per_label[sp[2]] += 1

    # Calculate P, R, F1 per class and take the unweighted macro mean
    p_scores, r_scores, f1_scores = [], [], []

    print("\n" + "="*65)
    print(f"{'LABEL':<12} | {'GOLD':<6} | {'PRED':<6} | {'PREC':<8} | {'RECALL':<8} | {'F1':<8}")
    print("="*65)

    for lbl in ALL_LABELS:
        tp = tp_per_label[lbl]
        fp = fp_per_label[lbl]
        fn = fn_per_label[lbl]
        
        total_gold = tp + fn
        total_pred = tp + fp

        p = tp / total_pred if total_pred > 0 else 0.0
        r = tp / total_gold if total_gold > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

        p_scores.append(p)
        r_scores.append(r)
        f1_scores.append(f1)

        print(f"{lbl:<12} | {total_gold:<6} | {total_pred:<6} | {p:.4f}   | {r:.4f}   | {f1:.4f}")

    macro_p = sum(p_scores) / len(p_scores)
    macro_r = sum(r_scores) / len(r_scores)
    macro_f1 = sum(f1_scores) / len(f1_scores)

    print("="*65)
    print(f"Strict Macro-Precision: {macro_p:.6f}")
    print(f"Strict Macro-Recall:    {macro_r:.6f}")
    print(f"Strict Macro-F1:        {macro_f1:.6f}")
    print("="*65)

    return macro_f1, macro_p, macro_r

In [70]:
model.eval()
val_predictions = {}

print("Evaluating locally on Fold 2...")
with torch.no_grad():
    for doc_id, text in val_texts.items():
        token_ids, offsets = tokenizer.encode_with_exact_offsets(text)
        if not token_ids:
            val_predictions[doc_id] = []
            continue

        doc_spans = []
        for start_idx in range(0, max(1, len(token_ids)), MAX_SEQ_LEN - STRIDE):
            end_idx = min(len(token_ids), start_idx + MAX_SEQ_LEN)
            chunk_ids = token_ids[start_idx:end_idx]
            chunk_offsets = offsets[start_idx:end_idx]

            input_tensor = torch.tensor([chunk_ids], dtype=torch.long, device=DEVICE)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(input_tensor)
            preds = torch.argmax(logits[0], dim=-1).cpu().tolist()

            spans = decode_spans_exact(text, preds, chunk_offsets)
            doc_spans.extend(spans)

        unique_spans = []
        for sp in doc_spans:
            if not any(sp[0] == s[0] and sp[1] == s[1] and sp[2] == s[2] for s in unique_spans):
                unique_spans.append([int(sp[0]), int(sp[1]), str(sp[2])])

        val_predictions[doc_id] = unique_spans

# Run the local strict scoring!
macro_f1, macro_p, macro_r = evaluate_predictions_strictly(val_golds, val_predictions)

Evaluating locally on Fold 2...

LABEL        | GOLD   | PRED   | PREC     | RECALL   | F1      
APP          | 200    | 301    | 0.2691   | 0.4050   | 0.3234
RESP         | 190    | 278    | 0.2878   | 0.4211   | 0.3419
A.COUNSEL    | 125    | 202    | 0.2079   | 0.3360   | 0.2569
R.COUNSEL    | 97     | 158    | 0.2215   | 0.3608   | 0.2745
JUDGE        | 122    | 172    | 0.1047   | 0.1475   | 0.1224
WIT          | 120    | 200    | 0.3750   | 0.6250   | 0.4688
AUTH         | 468    | 711    | 0.4149   | 0.6303   | 0.5004
COURT        | 558    | 748    | 0.5829   | 0.7814   | 0.6677
STAT         | 735    | 1204   | 0.4012   | 0.6571   | 0.4982
PREC         | 559    | 1025   | 0.1112   | 0.2039   | 0.1439
DATE         | 762    | 993    | 0.6405   | 0.8346   | 0.7248
CASENO       | 337    | 557    | 0.2352   | 0.3887   | 0.2931
Strict Macro-Precision: 0.320985
Strict Macro-Recall:    0.482631
Strict Macro-F1:        0.384660
